# RT Reviewer Fixes Runner

Closes the four reviewer gaps in the ACL Findings submission:

| # | Workload | Reviewer objection closed |
|---|---|---|
| A | LongLLMLingua baseline — hard stress set | "no strong NLP baseline" |
| B | Full LongMemEval-S (no 40-turn cap) | "truncated conversations mislead" |
| C | LongLLMLingua baseline — MSC valid | cross-benchmark baseline coverage |
| D | Llama-3.2-3B scale validation | "only one 3B model tested" |

**Resumable**: re-run any cell after a crash — completed shards are auto-skipped via `progress.json`.

**Cells run top-to-bottom, once.** Each section is independent after setup.

## 0 · Configuration — edit this cell first

In [ ]:
from pathlib import Path
import os

# ── repo ─────────────────────────────────────────────────────────────────────
REPO_URL   = "https://github.com/SteveMama/rt-geometry-memory.git"
REPO_DIR   = Path("/workspace/RT").resolve()   # change to your cloud mount if different

# ── credentials ──────────────────────────────────────────────────────────────
GITHUB_USER  = ""   # your GitHub username (for auto-push at the end)
GITHUB_TOKEN = ""   # personal access token with repo write scope
HF_TOKEN     = ""   # HuggingFace token — required only for Llama-3.2-3B (gated)

# ── experiment config ─────────────────────────────────────────────────────────
MODEL_KEY    = "qwen25_15b"          # primary model
BUDGETS      = "0.20,0.35,0.50"      # compression budgets
GPU_COUNT    = 0                      # 0 = all visible GPUs
JOB_MULTIPLIER = 2                    # shards per GPU
RUN_PREFIX   = "reviewer_fixes"

# ── workload toggles ──────────────────────────────────────────────────────────
RUN_HARDSET_BASELINES = True   # LongLLMLingua on hard stress set (Workload A)
RUN_FULL_LME          = True   # full LongMemEval-S no-truncation (Workload B)
RUN_MSC_BASELINES     = True   # LongLLMLingua on MSC valid (Workload C)
RUN_LLAMA32_3B        = True   # Llama-3.2-3B scale validation (Workload D)

# ── push toggles ──────────────────────────────────────────────────────────────
AUTO_PUSH   = True   # set False to skip the final git push

print(f"REPO_DIR  : {REPO_DIR}")
print(f"Model     : {MODEL_KEY}  Budgets: {BUDGETS}")
print(f"Workloads : hardset={RUN_HARDSET_BASELINES} fullLME={RUN_FULL_LME} "
      f"msc={RUN_MSC_BASELINES} llama32={RUN_LLAMA32_3B}")
print(f"Push      : {AUTO_PUSH}")

## 1 · Shell helper

In [ ]:
import subprocess, sys

def run(cmd, cwd=None, env=None, check=True):
    """Stream a shell command, raise on non-zero unless check=False."""
    print(f"\n$ {cmd}", flush=True)
    proc = subprocess.Popen(
        cmd, shell=True,
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    if check and proc.returncode != 0:
        raise RuntimeError(f"command failed ({proc.returncode}): {cmd}")
    return proc.returncode

def env_with(**extras):
    """Return os.environ merged with extras, filtering out None/empty values."""
    merged = {**os.environ}
    for k, v in extras.items():
        if v:
            merged[k] = str(v)
    return merged

print("helpers ready")

## 2 · Clone repo (skip if already present)

In [ ]:
if not REPO_DIR.exists():
    clone_url = REPO_URL
    if GITHUB_USER and GITHUB_TOKEN:
        proto, rest = REPO_URL.split("://", 1)
        clone_url = f"{proto}://{GITHUB_USER}:{GITHUB_TOKEN}@{rest}"
    run(f"git clone {clone_url} {REPO_DIR}")
else:
    print(f"{REPO_DIR} already exists, pulling latest")
    run("git pull --ff-only", cwd=REPO_DIR)

run("git log --oneline -5", cwd=REPO_DIR)

## 3 · Install dependencies

In [ ]:
VENV = REPO_DIR / ".venv"
PY   = str(VENV / "bin" / "python")

if not (VENV / "bin" / "python").exists():
    run(f"python3 -m venv {VENV}")

run(f"{PY} -m pip install -q --upgrade pip", cwd=REPO_DIR)

# install project + all reviewer deps in one shot
run(f"PYTHON_BIN={PY} bash scripts/install_reviewer_deps.sh", cwd=REPO_DIR)

## 4 · Verify GPU + torch

In [ ]:
run("nvidia-smi")
run(f"{PY} -c \""
    "import torch; "
    "print(f'torch={torch.__version__}'); "
    "print(f'cuda={torch.cuda.is_available()}'); "
    "print(f'gpus={torch.cuda.device_count()}'); "
    "[print(f'  [{i}] {torch.cuda.get_device_name(i)}') for i in range(torch.cuda.device_count())]"
    "\"", cwd=REPO_DIR)

# verify flash-attn (optional but fast)
run(f"{PY} -c \"import flash_attn; print('flash-attn', flash_attn.__version__)\"",
    check=False)

# verify llmlingua (required for longllmlingua baseline)
run(f"{PY} -c \"from llmlingua import PromptCompressor; print('llmlingua ok')\"")

## 5 · HuggingFace login (required for Llama-3.2-3B)

In [ ]:
if HF_TOKEN:
    run(f"{PY} -c \"from huggingface_hub import login; login('{HF_TOKEN}')\"")
    print("HuggingFace login ok")
else:
    print("HF_TOKEN not set — Llama-3.2-3B workload will be skipped")

## 6 · Download benchmarks

In [ ]:
# ── MSC valid ─────────────────────────────────────────────────────────────────
BENCH = REPO_DIR / "benchmarks"
BENCH.mkdir(parents=True, exist_ok=True)

MSC_RAW   = BENCH / "msc_valid_raw.jsonl"
MSC_JSONL = BENCH / "msc_valid_normalized.jsonl"

if not MSC_JSONL.exists():
    run(f"{PY} scripts/download_public_benchmark.py "
        f"--benchmark msc_valid --output {MSC_RAW}", cwd=REPO_DIR)
    run(f"{PY} scripts/prepare_public_benchmark_jsonl.py "
        f"--format msc --input {MSC_RAW} --output {MSC_JSONL} "
        f"--family msc_valid", cwd=REPO_DIR)
else:
    print(f"MSC valid already present: {MSC_JSONL}")

import json
n_msc = sum(1 for _ in open(MSC_JSONL))
print(f"MSC valid: {n_msc} conversations")

In [ ]:
# ── LongMemEval-S FULL (no turn cap — fixes the key reviewer objection) ────────
LME_RAW   = BENCH / "longmemeval_s_raw.json"
LME_JSONL = BENCH / "longmemeval_s_full_normalized.jsonl"   # different filename from the truncated version

if not LME_JSONL.exists():
    run(f"{PY} scripts/download_public_benchmark.py "
        f"--benchmark longmemeval_s_cleaned --output {LME_RAW}", cwd=REPO_DIR)
    # NOTE: no --max-turns-per-conversation flag → full conversations
    run(f"{PY} scripts/prepare_public_benchmark_jsonl.py "
        f"--format longmemeval --input {LME_RAW} --output {LME_JSONL} "
        f"--family longmemeval_s_full", cwd=REPO_DIR)
else:
    print(f"LongMemEval-S full already present: {LME_JSONL}")

n_lme = sum(1 for _ in open(LME_JSONL))
print(f"LongMemEval-S full: {n_lme} conversations")

In [ ]:
# ── Hard stress set (already in repo) ─────────────────────────────────────────
HARDSET = REPO_DIR / "paper1_geometry" / "assets" / "paper2_behavior_stress_conversations.jsonl"
assert HARDSET.exists(), f"hard stress set missing: {HARDSET}"
n_hs = sum(1 for _ in open(HARDSET))
print(f"Hard stress set: {n_hs} conversations  ({HARDSET})")

## 7 · Build shared environment for GPU workers

In [ ]:
import subprocess

# count GPUs
result = subprocess.run(
    "nvidia-smi --query-gpu=index --format=csv,noheader",
    shell=True, capture_output=True, text=True
)
all_gpu_ids = [g.strip() for g in result.stdout.strip().splitlines() if g.strip()]
n_gpus = GPU_COUNT if GPU_COUNT > 0 else len(all_gpu_ids)
gpu_ids = all_gpu_ids[:n_gpus]

JOB_SHARDS = n_gpus * JOB_MULTIPLIER

SHARED_ENV = env_with(
    PYTHON_BIN=PY,
    MODEL_KEY=MODEL_KEY,
    BUDGETS=BUDGETS,
    GPU_COUNT=str(n_gpus),
    JOB_MULTIPLIER=str(JOB_MULTIPLIER),
    RUN_PREFIX=RUN_PREFIX,
    HF_TOKEN=HF_TOKEN,
    HUGGING_FACE_HUB_TOKEN=HF_TOKEN,
    TOKENIZERS_PARALLELISM="false",
    PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True",
    GITHUB_USER=GITHUB_USER,
    GITHUB_TOKEN=GITHUB_TOKEN,
    SKIP_DOWNLOAD="1",   # already done above
    SKIP_PUSH="1",       # we'll push manually at the end
)

print(f"GPUs detected : {all_gpu_ids}")
print(f"GPUs used     : {gpu_ids}")
print(f"Shards/workload: {JOB_SHARDS}")

---
## Workload A · LongLLMLingua baseline — hard stress set
Closes: *"no strong NLP compression baseline"*

In [ ]:
if RUN_HARDSET_BASELINES:
    OUT_A = REPO_DIR / "results" / "reviewer_fixes" / "baselines"
    OUT_A.mkdir(parents=True, exist_ok=True)
    PLAN_A = REPO_DIR / "results" / "reviewer_fixes" / "shard_plans" / "baselines_hardset"
    PLAN_A.mkdir(parents=True, exist_ok=True)

    run(
        f"{PY} -m paper3_codec.plan_conversation_shards "
        f"--input-path {HARDSET} "
        f"--shard-count {JOB_SHARDS} "
        f"--target-turn-stride 1 "
        f"--output-dir {PLAN_A}",
        cwd=REPO_DIR
    )
    print(f"Planned {JOB_SHARDS} shards → {PLAN_A}")
else:
    print("RUN_HARDSET_BASELINES=False, skipping")

In [ ]:
if RUN_HARDSET_BASELINES:
    env_a = {**SHARED_ENV,
             "HARDSET_INPUT": str(HARDSET),
             "PLAN_DIR_HS": str(PLAN_A),
             "JOB_SHARDS": str(JOB_SHARDS)}

    # write a small orchestrator fragment that the sh script sources
    run(
        "bash -c '"
        f"set -euo pipefail; "
        f"cd {REPO_DIR}; "
        f"SKIP_DOWNLOAD=1 SKIP_PUSH=1 "
        f"RUN_HARDSET_BASELINES=1 RUN_FULL_LME=0 RUN_MSC_BASELINES=0 RUN_LLAMA32_3B=0 "
        f"HARDSET_INPUT={HARDSET} MSC_INPUT={MSC_JSONL} LONGMEM_INPUT={LME_JSONL} "
        f"PYTHON_BIN={PY} GPU_COUNT={n_gpus} JOB_MULTIPLIER={JOB_MULTIPLIER} "
        f"MODEL_KEY={MODEL_KEY} BUDGETS={BUDGETS} RUN_PREFIX={RUN_PREFIX} "
        "bash scripts/run_reviewer_fixes_multigpu.sh'",
        cwd=REPO_DIR, env=env_a
    )
    print("Workload A complete")

In [ ]:
if RUN_HARDSET_BASELINES:
    csvs = list((REPO_DIR / "results" / "reviewer_fixes" / "baselines").rglob("evaluation_rows.csv"))
    print(f"Workload A outputs: {len(csvs)} evaluation CSV(s)")
    for c in sorted(csvs):
        print(f"  {c.relative_to(REPO_DIR)}")

---
## Workload B · Full LongMemEval-S (no 40-turn cap)
Closes: *"40-turn truncation of 400-600 turn conversations is misleading"*

In [ ]:
if RUN_FULL_LME:
    OUT_B = REPO_DIR / "results" / "reviewer_fixes" / "fullLME"
    OUT_B.mkdir(parents=True, exist_ok=True)
    PLAN_B = REPO_DIR / "results" / "reviewer_fixes" / "shard_plans" / "fullLME"
    PLAN_B.mkdir(parents=True, exist_ok=True)

    run(
        f"{PY} -m paper3_codec.plan_conversation_shards "
        f"--input-path {LME_JSONL} "
        f"--shard-count {JOB_SHARDS} "
        f"--target-turn-stride 1 "
        f"--output-dir {PLAN_B}",
        cwd=REPO_DIR
    )
    print(f"Planned {JOB_SHARDS} shards → {PLAN_B}")
else:
    print("RUN_FULL_LME=False, skipping")

In [ ]:
if RUN_FULL_LME:
    env_b = {**SHARED_ENV,
             "HARDSET_INPUT": str(HARDSET),
             "MSC_INPUT": str(MSC_JSONL),
             "LONGMEM_INPUT": str(LME_JSONL)}

    run(
        "bash -c '"
        f"set -euo pipefail; "
        f"cd {REPO_DIR}; "
        f"SKIP_DOWNLOAD=1 SKIP_PUSH=1 "
        f"RUN_HARDSET_BASELINES=0 RUN_FULL_LME=1 RUN_MSC_BASELINES=0 RUN_LLAMA32_3B=0 "
        f"HARDSET_INPUT={HARDSET} MSC_INPUT={MSC_JSONL} LONGMEM_INPUT={LME_JSONL} "
        f"PYTHON_BIN={PY} GPU_COUNT={n_gpus} JOB_MULTIPLIER={JOB_MULTIPLIER} "
        f"MODEL_KEY={MODEL_KEY} BUDGETS={BUDGETS} RUN_PREFIX={RUN_PREFIX} "
        "bash scripts/run_reviewer_fixes_multigpu.sh'",
        cwd=REPO_DIR, env=env_b
    )
    print("Workload B complete")

In [ ]:
if RUN_FULL_LME:
    csvs = list((REPO_DIR / "results" / "reviewer_fixes" / "fullLME").rglob("evaluation_rows.csv"))
    print(f"Workload B outputs: {len(csvs)} evaluation CSV(s)")
    for c in sorted(csvs):
        print(f"  {c.relative_to(REPO_DIR)}")

---
## Workload C · LongLLMLingua baseline — MSC valid
Cross-benchmark baseline coverage

In [ ]:
if RUN_MSC_BASELINES:
    PLAN_C = REPO_DIR / "results" / "reviewer_fixes" / "shard_plans" / "baselines_msc"
    PLAN_C.mkdir(parents=True, exist_ok=True)

    run(
        f"{PY} -m paper3_codec.plan_conversation_shards "
        f"--input-path {MSC_JSONL} "
        f"--shard-count {JOB_SHARDS} "
        f"--target-turn-stride 1 "
        f"--output-dir {PLAN_C}",
        cwd=REPO_DIR
    )
    print(f"Planned {JOB_SHARDS} shards → {PLAN_C}")
else:
    print("RUN_MSC_BASELINES=False, skipping")

In [ ]:
if RUN_MSC_BASELINES:
    env_c = {**SHARED_ENV,
             "HARDSET_INPUT": str(HARDSET),
             "MSC_INPUT": str(MSC_JSONL),
             "LONGMEM_INPUT": str(LME_JSONL)}

    run(
        "bash -c '"
        f"set -euo pipefail; "
        f"cd {REPO_DIR}; "
        f"SKIP_DOWNLOAD=1 SKIP_PUSH=1 "
        f"RUN_HARDSET_BASELINES=0 RUN_FULL_LME=0 RUN_MSC_BASELINES=1 RUN_LLAMA32_3B=0 "
        f"HARDSET_INPUT={HARDSET} MSC_INPUT={MSC_JSONL} LONGMEM_INPUT={LME_JSONL} "
        f"PYTHON_BIN={PY} GPU_COUNT={n_gpus} JOB_MULTIPLIER={JOB_MULTIPLIER} "
        f"MODEL_KEY={MODEL_KEY} BUDGETS={BUDGETS} RUN_PREFIX={RUN_PREFIX} "
        "bash scripts/run_reviewer_fixes_multigpu.sh'",
        cwd=REPO_DIR, env=env_c
    )
    print("Workload C complete")

In [ ]:
if RUN_MSC_BASELINES:
    csvs = list((REPO_DIR / "results" / "reviewer_fixes" / "baselines").rglob("*msc*evaluation_rows.csv"))
    print(f"Workload C outputs: {len(csvs)} evaluation CSV(s)")
    for c in sorted(csvs):
        print(f"  {c.relative_to(REPO_DIR)}")

---
## Workload D · Llama-3.2-3B scale validation
Closes: *"only one 3B model tested"* — requires HF_TOKEN

In [ ]:
if RUN_LLAMA32_3B:
    if not HF_TOKEN:
        print("WARNING: HF_TOKEN not set — skipping Llama-3.2-3B (gated model)")
    else:
        env_d = {**SHARED_ENV,
                 "HARDSET_INPUT": str(HARDSET),
                 "MSC_INPUT": str(MSC_JSONL),
                 "LONGMEM_INPUT": str(LME_JSONL)}

        run(
            "bash -c '"
            f"set -euo pipefail; "
            f"cd {REPO_DIR}; "
            f"SKIP_DOWNLOAD=1 SKIP_PUSH=1 "
            f"RUN_HARDSET_BASELINES=0 RUN_FULL_LME=0 RUN_MSC_BASELINES=0 RUN_LLAMA32_3B=1 "
            f"HARDSET_INPUT={HARDSET} MSC_INPUT={MSC_JSONL} LONGMEM_INPUT={LME_JSONL} "
            f"PYTHON_BIN={PY} GPU_COUNT={n_gpus} JOB_MULTIPLIER={JOB_MULTIPLIER} "
            f"MODEL_KEY={MODEL_KEY} BUDGETS={BUDGETS} RUN_PREFIX={RUN_PREFIX} "
            f"HF_TOKEN={HF_TOKEN} HUGGING_FACE_HUB_TOKEN={HF_TOKEN} "
            "bash scripts/run_reviewer_fixes_multigpu.sh'",
            cwd=REPO_DIR, env=env_d
        )
        print("Workload D complete")
else:
    print("RUN_LLAMA32_3B=False, skipping")

In [ ]:
if RUN_LLAMA32_3B and HF_TOKEN:
    csvs = list((REPO_DIR / "results" / "reviewer_fixes" / "scale_llama32_3b").rglob("evaluation_rows.csv"))
    print(f"Workload D outputs: {len(csvs)} evaluation CSV(s)")
    for c in sorted(csvs):
        print(f"  {c.relative_to(REPO_DIR)}")

---
## 8 · Inspect results inline

In [ ]:
import pandas as pd

results_root = REPO_DIR / "results" / "reviewer_fixes"
all_csvs = sorted(results_root.rglob("evaluation_rows.csv"))
print(f"Total evaluation CSVs: {len(all_csvs)}")
for c in all_csvs:
    df = pd.read_csv(c)
    print(f"\n{'='*60}")
    print(c.relative_to(REPO_DIR))
    print(f"  rows={len(df)}  cols={list(df.columns)}")
    if 'policy' in df.columns:
        print(df.groupby('policy')['score'].describe().round(3).to_string())

In [ ]:
# print any markdown reports generated
for report in sorted(results_root.rglob("*report*.md")):
    print(f"\n{'='*80}")
    print(report.relative_to(REPO_DIR))
    print('='*80)
    print(report.read_text()[:3000])

In [ ]:
# print the auto-generated run summary
summaries = sorted(results_root.glob(f"summary_{RUN_PREFIX}*.md"))
for s in summaries:
    print(s.read_text())

---
## 9 · Commit & push to GitHub

In [ ]:
if not AUTO_PUSH:
    print("AUTO_PUSH=False — skipping git push")
elif not GITHUB_USER or not GITHUB_TOKEN:
    print("ERROR: Set GITHUB_USER and GITHUB_TOKEN in cell 0 to enable push")
else:
    import datetime
    run(
        f"GITHUB_USER={GITHUB_USER} GITHUB_TOKEN={GITHUB_TOKEN} "
        f"bash scripts/colab_commit_push.sh "
        f"SteveMama pranav@vizit.com "
        f"'reviewer-fix results: longllmlingua baseline, full LME, llama32_3b [{datetime.date.today()}]' "
        f"results/reviewer_fixes "
        f"june_fixes/baselines/baseline_study.py "
        f"paper1_geometry/modeling.py "
        f"scripts/install_reviewer_deps.sh "
        f"scripts/run_reviewer_fixes_multigpu.sh "
        f"notebooks/rt_reviewer_fixes_runner.ipynb",
        cwd=REPO_DIR
    )
    print("Pushed to GitHub ✓")

## 10 · Package results for download (optional)

In [ ]:
import shutil

archive = shutil.make_archive(
    str(REPO_DIR / f"reviewer_fixes_results_{RUN_PREFIX}"),
    "zip",
    root_dir=str(REPO_DIR / "results"),
    base_dir="reviewer_fixes",
)
size_mb = Path(archive).stat().st_size / 1e6
print(f"Archive: {archive}  ({size_mb:.1f} MB)")
print("Download this file from your cloud file manager or scp it out.")